# Entraînement YOLO compost sur Colab (GPU)

**Règle d'or : le code se modifie dans le repo et se commit, JAMAIS dans ce notebook.**
Ce notebook ne fait qu'orchestrer : clone, install, données, scripts, sauvegarde.
Colab est en LECTURE SEULE vis-à-vis de git : on clone, aucune cellule ne
commit ni ne push — rien de ce qui se passe ici n'apparaît sur GitHub.

Prérequis :
- runtime GPU (Exécution > Modifier le type d'exécution > T4 GPU) ;
- un token GitHub personnel classique (scope `repo`) dans les Secrets Colab
  sous `GITHUB_TOKEN` — il ne sert qu'au clone ;
- le(s) dataset(s) zippé(s) sur Drive, prêts pour `prepare_dataset.py` :
  `MyDrive/compost/dataset_raw.zip` (+ `dataset_raw_<nom>.zip` par dataset
  supplémentaire) : `images/` + `labels/` + `groups.csv` (sortie de
  `import_dataset.py`), ou les fichiers `cap_*` d'un export de l'interface.

In [ ]:
# 1. Clone du repo (token lu depuis les Secrets Colab — utilisé uniquement pour cloner)
BRANCH = 'yolo'   # branche de travail ; mettre 'main' après fusion
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone --depth 1 --branch {BRANCH} https://{token}@github.com/TSResearch-hub/Compost_Waste_Yolo.git /content/repo
# le code d'entraînement est le sous-dossier compost-yolo du repo
%cd /content/repo/compost-yolo

In [ ]:
# 2. Installation des dépendances
!pip install -q -e .

In [ ]:
# 3. Montage de Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Copie + dézippage du dataset vers le disque local de Colab
# (ne JAMAIS s'entraîner directement sur le dossier Drive monté : trop lent)
!cp /content/drive/MyDrive/compost/dataset_raw.zip /content/
!unzip -q -o /content/dataset_raw.zip -d /content/dataset_raw

In [ ]:
# 5. Préparation du dataset (split par session)
!python scripts/prepare_dataset.py --source /content/dataset_raw --output /content/dataset

In [ ]:
# 5b. (optionnel) Datasets supplémentaires : ils s'ACCUMULENT dans /content/dataset
# (le split par hash garantit que les images déjà préparées ne changent pas de split)
for name in []:  # ex. : for name in ['taco']:
    !cp /content/drive/MyDrive/compost/dataset_raw_{name}.zip /content/
    !unzip -q -o /content/dataset_raw_{name}.zip -d /content/dataset_raw_{name}
    !python scripts/prepare_dataset.py --source /content/dataset_raw_{name} --output /content/dataset

In [ ]:
# 5c. Histogramme des instances par classe et par split (contrôle après tous les imports)
import yaml
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

names = yaml.safe_load(open('configs/data.yaml'))['names']
splits = ['train', 'val', 'test']
counts = {s: Counter() for s in splits}
for s in splits:
    for label_file in Path(f'/content/dataset/labels/{s}').glob('*.txt'):
        for line in label_file.read_text().splitlines():
            if line.strip():
                counts[s][int(line.split()[0])] += 1

x = range(len(names))
width = 0.27
plt.figure(figsize=(10, 4))
for i, s in enumerate(splits):
    plt.bar([v + (i - 1) * width for v in x], [counts[s][j] for j in x], width, label=s)
plt.yscale('log')  # échelle log : sans elle, les classes rares sont invisibles
plt.xticks(list(x), names, rotation=20)
plt.ylabel('instances (échelle log)')
plt.title('Instances par classe et par split')
plt.legend(); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.show()
for s in splits:
    print(f"{s}: {sum(counts[s].values())} instances —",
          ", ".join(f"{names[j]}: {counts[s][j]}" for j in x))

In [ ]:
# 6. Entraînement (checkpoints sauvegardés sur Drive toutes les 10 epochs)
# Reprise après coupure : ajouter --resume /content/runs/train_xxx/weights/last.pt
!python scripts/train.py --data /content/dataset/data.yaml \
    --runs-dir /content/runs \
    --backup-dir /content/drive/MyDrive/compost/backups --backup-every 10

In [ ]:
# 7. Copie du run complet (poids + métriques) vers Drive
!mkdir -p /content/drive/MyDrive/compost/runs
!cp -r /content/runs/* /content/drive/MyDrive/compost/runs/
!ls /content/drive/MyDrive/compost/runs